# Arrow interoperability - Python

All 3 Python examples from [docs/arrow.md](https://platob.github.io/yggdryl/arrow/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
import pyarrow as pa
from yggdryl import Field

field = Field("symbol", "utf8", nullable=False)
scalar = field.default_arrow_scalar()

assert isinstance(scalar, pa.Scalar)
assert scalar.type == pa.string()
assert scalar.as_py() == ""

## Nullability picks the default

In [ ]:
from yggdryl import DataType, Field

assert DataType("int64").default_arrow_scalar().as_py() == 0

# A field is nullable unless you say otherwise, and its default is null.
optional = Field("symbol", "utf8").default_arrow_scalar()
assert not optional.is_valid
assert optional.as_py() is None

try:
    Field("never", "null", nullable=False).default_arrow_scalar()
except ValueError as error:
    assert "logical-null default" in str(error)
else:
    raise AssertionError("a required null column has no default")

## A struct root is one row

In [ ]:
from yggdryl import DataType, Field

schema = Field(
    "row",
    DataType.from_fields(
        [Field("id", "int64", nullable=False), Field("symbol", "utf8")]
    ),
    nullable=False,
)

scalar = schema.default_arrow_scalar()
assert scalar.as_py() == {"id": 0, "symbol": None}
assert scalar["id"].as_py() == 0